In [ ]:
import json
import pandas as pd
import numpy as np

In [ ]:
with open("../data/processed/moves_with_coordinates.json") as f:
    moves = json.load(f)

df = pd.DataFrame(moves)
df.head()

In [ ]:
df = df.sort_values("order").reset_index(drop=True)
df[["order", "address", "area", "year", "month"]]

In [ ]:
df["month"] = df["month"].fillna(7)

df["move_date"] = pd.to_datetime(
    dict(year=df.year, month=df.month, day=1)
)

df[["order", "address", "move_date"]]

In [ ]:
df["months_since_previous_move"] = (
    df["move_date"].diff().dt.days / 30
)

df[["address", "months_since_previous_move"]]

In [ ]:
df["estimated_value_mid_kr"] = (
    df["estimated_value_low_kr"] +
    df["estimated_value_high_kr"]
) / 2

In [ ]:
import math

def haversine(lat1, lon1, lat2, lon2):
    """
    Calculate distance between two coordinates in kilometers.
    """
    R = 6371  # Earth radius in km
    
    lat1 = math.radians(lat1)
    lon1 = math.radians(lon1)
    lat2 = math.radians(lat2)
    lon2 = math.radians(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )

    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c

In [ ]:
distances = [None]

for i in range(1, len(df)):
    
    lat1 = df.loc[i-1, "latitude"]
    lon1 = df.loc[i-1, "longitude"]
    
    lat2 = df.loc[i, "latitude"]
    lon2 = df.loc[i, "longitude"]
    
    d = haversine(lat1, lon1, lat2, lon2)
    
    distances.append(d)

df["distance_from_previous_km"] = distances

df[["order", "address", "distance_from_previous_km"]]

In [ ]:
df.plot(
    x="distance_from_previous_km",
    y="months_since_previous_move",
    kind="scatter"
)

In [ ]:
!pip install folium

In [ ]:
import folium

In [ ]:
center_lat = df["latitude"].mean()
center_lon = df["longitude"].mean()

m = folium.Map(location=[center_lat, center_lon], zoom_start=10)
m

In [ ]:
for _, row in df.iterrows():
    popup_text = (
        f"Order: {row['order']}<br>"
        f"Address: {row['address']}<br>"
        f"Area: {row['area']}<br>"
        f"Date: {row['move_date'].strftime('%Y-%m')}"
    )
    
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=popup_text,
        tooltip=f"{row['order']}: {row['address']}"
    ).add_to(m)

m

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

for _, row in df.iterrows():
    popup_text = (
        f"Order: {row['order']}<br>"
        f"Address: {row['address']}<br>"
        f"Area: {row['area']}<br>"
        f"Date: {row['move_date'].strftime('%Y-%m')}"
    )
    
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=popup_text,
        tooltip=f"{row['order']}: {row['address']}"
    ).add_to(m)

coordinates = df[["latitude", "longitude"]].values.tolist()

folium.PolyLine(
    coordinates,
    color="blue",
    weight=3,
    opacity=0.7
).add_to(m)

m

In [ ]:
m = folium.Map(location=[center_lat, center_lon], zoom_start=10)

for _, row in df.iterrows():
    folium.Marker(
        location=[row["latitude"], row["longitude"]],
        popup=(
            f"Order: {row['order']}<br>"
            f"Address: {row['address']}<br>"
            f"Area: {row['area']}<br>"
            f"Date: {row['move_date'].strftime('%Y-%m')}"
        ),
        tooltip=f"{row['order']}: {row['address']}",
        icon=folium.DivIcon(
            html=f"""
            <div style="font-size: 12pt; color: black; font-weight: bold">
                {row['order']}
            </div>
            """
        )
    ).add_to(m)

folium.PolyLine(
    df[["latitude", "longitude"]].values.tolist(),
    color="blue",
    weight=3,
    opacity=0.7
).add_to(m)

m

In [ ]:
m.save("../data/processed/move_history_map.html")

In [ ]:
df["months_until_next_move"] = df["months_since_previous_move"].shift(-1)
df

In [ ]:
train_df = df.dropna(subset=["months_until_next_move"])

In [ ]:
features = [
    "distance_from_previous_km",
    "living_area_m2",
    "estimated_value_mid_kr"
]

X = train_df[features].fillna(0)
y = train_df["months_until_next_move"]

In [ ]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()

regressor.fit(X, y)

In [ ]:
pd.DataFrame({
    "feature": features,
    "coefficient": regressor.coef_
})

In [ ]:
latest_home = df.iloc[-1]

X_next = pd.DataFrame([[
    latest_home["distance_from_previous_km"],
    latest_home["living_area_m2"],
    latest_home["estimated_value_mid_kr"]
]], columns=features)

predicted_months = regressor.predict(X_next)

predicted_months

In [ ]:
df["next_latitude"] = df["latitude"].shift(-1)
df["next_longitude"] = df["longitude"].shift(-1)

df[["order","latitude","next_latitude","longitude","next_longitude"]]

In [ ]:
train_df = df.dropna(subset=["next_latitude", "next_longitude"])

In [ ]:
features = [
    "latitude",
    "longitude",
    "distance_from_previous_km",
    "living_area_m2",
    "estimated_value_mid_kr"
]

X = train_df[features].fillna(0)

y_lat = train_df["next_latitude"]
y_lon = train_df["next_longitude"]

In [ ]:
from sklearn.linear_model import LinearRegression

lat_model = LinearRegression()
lon_model = LinearRegression()

lat_model.fit(X, y_lat)
lon_model.fit(X, y_lon)

In [ ]:
latest = df.iloc[-1]

X_next = pd.DataFrame([[
    latest["latitude"],
    latest["longitude"],
    latest["distance_from_previous_km"],
    latest["living_area_m2"],
    latest["estimated_value_mid_kr"]
]], columns=features)

pred_lat = lat_model.predict(X_next)[0]
pred_lon = lon_model.predict(X_next)[0]

pred_lat, pred_lon

In [ ]:
folium.Marker(
    location=[pred_lat, pred_lon],
    popup="Predicted next move",
    tooltip="Predicted move",
    icon=folium.Icon(color="red")
).add_to(m)

m

In [ ]:
from datetime import datetime

pred_lat = float(pred_lat)
pred_lon = float(pred_lon)

predicted_next_move = datetime(2026, 12, 1)

swedish_months = {
    1: "januari",
    2: "februari",
    3: "mars",
    4: "april",
    5: "maj",
    6: "juni",
    7: "juli",
    8: "augusti",
    9: "september",
    10: "oktober",
    11: "november",
    12: "december"
}

predicted_move_label_sv = f"{swedish_months[predicted_next_move.month]} {predicted_next_move.year}"

prediction = {
    "predicted_move_date": predicted_next_move.strftime("%Y-%m"),
    "predicted_move_label_sv": predicted_move_label_sv,
    "predicted_latitude": pred_lat,
    "predicted_longitude": pred_lon,
    "prediction_method": "average_move_gap_and_coordinate_regression"
}

with open("../data/processed/next_move_prediction.json", "w", encoding="utf-8") as f:
    json.dump(prediction, f, ensure_ascii=False, indent=2)